# 31 — Second-Order Greeks & Hessians

Full second-order risk with `jax.hessian` — Gamma, Vanna, Volga, and more.

| Greek | Definition | Indices |
|-------|-----------|--------|
| Gamma | ∂²V/∂S² | (S, S) |
| Vanna | ∂²V/∂S∂σ | (S, σ) |
| Volga | ∂²V/∂σ² | (σ, σ) |
| Charm | ∂²V/∂S∂T | (S, T) |
| Veta  | ∂²V/∂σ∂T | (σ, T) |

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.engines.analytic.heston import heston_price

---
## 1. Full Hessian Matrix

In [ ]:
S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.03, 0.01, 0.2

def bsm_vec(params):
    return bsm_price(params[0], params[1], params[2], params[3], params[4], params[5], 1.0)

params = jnp.array([S, K, T, r, q, sigma])
H = jax.hessian(bsm_vec)(params)

labels = ['S', 'K', 'T', 'r', 'q', 'σ']

plt.figure(figsize=(7, 6))
plt.imshow(np.array(H), cmap='RdBu_r', aspect='auto')
plt.xticks(range(6), labels)
plt.yticks(range(6), labels)
plt.colorbar(label='∂²V / ∂xᵢ∂xⱼ')
plt.title('BSM Full Hessian')
for i in range(6):
    for j in range(6):
        val = float(H[i, j])
        if abs(val) > 0.01:
            plt.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=7)
plt.show()

In [ ]:
# Named second-order Greeks
gamma = float(H[0, 0])  # ∂²V/∂S²
vanna = float(H[0, 5])  # ∂²V/∂S∂σ
volga = float(H[5, 5])  # ∂²V/∂σ²
charm = float(H[0, 2])  # ∂²V/∂S∂T
veta  = float(H[5, 2])  # ∂²V/∂σ∂T

print(f"Gamma (∂²V/∂S²)  : {gamma:.8f}")
print(f"Vanna (∂²V/∂S∂σ) : {vanna:.8f}")
print(f"Volga (∂²V/∂σ²)  : {volga:.8f}")
print(f"Charm (∂²V/∂S∂T) : {charm:.8f}")
print(f"Veta  (∂²V/∂σ∂T) : {veta:.8f}")

# Verify symmetry
print(f"\nHessian symmetric: {bool(jnp.allclose(H, H.T, atol=1e-12))}")

---
## 2. Gamma Surface

In [ ]:
strikes = jnp.linspace(70, 130, 60)
maturities = jnp.linspace(0.05, 2.0, 50)
KK, TT = jnp.meshgrid(strikes, maturities)

gamma_fn = jax.jit(jax.vmap(jax.vmap(
    lambda K, T: jax.grad(jax.grad(bsm_price, argnums=0), argnums=0)(S, K, T, r, q, sigma, 1.0),
    in_axes=(0, 0)), in_axes=(0, 0)))

G = gamma_fn(KK, TT)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(np.array(KK), np.array(TT), np.array(G), cmap='viridis', alpha=0.8)
ax.set_xlabel('Strike')
ax.set_ylabel('Maturity')
ax.set_zlabel('Gamma')
ax.set_title('BSM Gamma Surface')
plt.show()

---
## 3. Heston Hessian

In [ ]:
v0_h, kappa_h, theta_h, xi_h, rho_h = 0.04, 1.5, 0.04, 0.3, -0.7

def heston_vec(params):
    return heston_price(params[0], params[1], params[2], params[3], params[4],
                        params[5], params[6], params[7], params[8], params[9], 1)

heston_params = jnp.array([S, K, T, r, q, v0_h, kappa_h, theta_h, xi_h, rho_h])
H_heston = jax.hessian(heston_vec)(heston_params)

heston_labels = ['S', 'K', 'T', 'r', 'q', 'v₀', 'κ', 'θ', 'ξ', 'ρ']

plt.figure(figsize=(10, 8))
plt.imshow(np.array(H_heston), cmap='RdBu_r', aspect='auto')
plt.xticks(range(10), heston_labels, fontsize=9)
plt.yticks(range(10), heston_labels, fontsize=9)
plt.colorbar(label='∂²V / ∂xᵢ∂xⱼ')
plt.title('Heston Full Hessian (10×10)')
plt.show()

print(f"Heston Gamma (∂²V/∂S²)   : {float(H_heston[0, 0]):.8f}")
print(f"Heston ∂²V/∂S∂v₀         : {float(H_heston[0, 5]):.8f}")
print(f"Heston ∂²V/∂ξ∂ρ          : {float(H_heston[8, 9]):.8f}")

---
## 4. Third-Order: Speed of Gamma

In [ ]:
# Third derivative: ∂³V/∂S³ = d(Gamma)/dS = "Speed"
speed = jax.grad(jax.grad(jax.grad(bsm_price, argnums=0), argnums=0), argnums=0)

speed_val = float(speed(S, K, T, r, q, sigma, 1.0))
print(f"Speed (∂³V/∂S³) = {speed_val:.10f}")

# Speed across strikes
speed_batch = jax.jit(jax.vmap(lambda K: speed(S, K, T, r, q, sigma, 1.0)))
speed_vals = speed_batch(strikes)

plt.figure(figsize=(8, 5))
plt.plot(np.array(strikes), np.array(speed_vals))
plt.xlabel('Strike')
plt.ylabel('Speed (∂³V/∂S³)')
plt.title('Third-Order Greek: Speed')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', lw=0.5)
plt.show()

---
## 5. Performance: Hessian vs Repeated grad

In [ ]:
hess_fn = jax.jit(jax.hessian(bsm_vec))
_ = hess_fn(params)  # warm

# Individual second derivatives
def manual_hessian(p):
    g00 = jax.grad(jax.grad(bsm_vec))(p)[0]
    g05 = jax.grad(lambda x: jax.grad(bsm_vec)(x)[0])(p)[5]
    g55 = jax.grad(jax.grad(bsm_vec))(p)[5]
    return g00, g05, g55

manual_jit = jax.jit(manual_hessian)
_ = manual_jit(params)  # warm

ms_full, _ = timed_ms(hess_fn, params)
ms_manual, _ = timed_ms(manual_jit, params)

print(f"Full 6×6 Hessian        : {ms_full:.3f} ms")
print(f"3 individual 2nd derivs : {ms_manual:.3f} ms")
print(f"Hessian gives 36 values at {ms_full / 36:.4f} ms each")

---
## 6. Exercises

1. **Portfolio Gamma**: Compute aggregate Gamma for a 1,000-option portfolio using `vmap(hessian)`.
2. **Colour**: Compute ∂Gamma/∂T = ∂³V/∂S²∂T for BSM puts.
3. **Heston cross-gammas**: Plot ∂²V/∂S∂ρ across the strike range.

---
*End of Notebook 31*